This notebook is intended to show the process we require to do if we are taking our model to production <br> 
with out using pipelines.


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,OrdinalEncoder

In [47]:
heart = pd.read_csv('heart_disease.csv')
heart = heart[['Sex','Age','ChestPainType','Severity','ST_Slope','RestingECG','HeartDisease']]
heart.shape


(918, 7)

In [25]:
X_train,X_test,y_train,y_test = train_test_split(heart.drop(columns=["HeartDisease"]),heart['HeartDisease'],test_size=0.3,random_state=42)

In [32]:
heart['ChestPainType'].value_counts()

ChestPainType
ASY    496
NAP    203
ATA    173
TA      46
Name: count, dtype: int64

In [36]:
# encoding ordinal columns : 
oe = OrdinalEncoder(categories=[['Down','Flat','Up'],['Low','Medium','High','Critical']],dtype=int)
X_train_ordinal_cols = oe.fit_transform(X_train[['ST_Slope','Severity']])
X_test_ordinal_cols = oe.transform(X_test[['ST_Slope','Severity']])
X_train_ordinal_cols



array([[2, 0],
       [0, 1],
       [1, 0],
       ...,
       [2, 1],
       [2, 1],
       [1, 1]])

In [35]:
# encoding nominal cols
ohe = OneHotEncoder(drop='first',sparse_output=False,dtype=int)
X_train_nominal_cols = ohe.fit_transform(X_train[['Sex','ChestPainType','RestingECG']])
X_test_nominal_cols= ohe.transform(X_test[['Sex','ChestPainType','RestingECG']])
X_test_nominal_cols

array([[0, 1, 0, 0, 1, 0],
       [1, 0, 1, 0, 1, 0],
       [1, 0, 0, 0, 0, 1],
       ...,
       [1, 0, 0, 0, 0, 0],
       [1, 0, 1, 0, 1, 0],
       [0, 0, 1, 0, 1, 0]])

In [46]:
# since there is no missing values , we dont need to impute. 
# we can extract remainng cols or since there is only one column left 'Age' column. so we simply  add it.
X_train_Age = X_train[['Age']]
X_test_Age = X_test[['Age']]
# so addding all together
X_train_transformed = np.concatenate((X_train_Age,X_train_ordinal_cols,X_train_nominal_cols),axis=1)
X_test_transformed = np.concatenate((X_test_Age,X_test_ordinal_cols,X_test_nominal_cols),axis=1)
X_train_transformed.shape


(642, 9)

In [48]:
# encoding target variable
le = LabelEncoder()
y_train_transformed = le.fit_transform(y_train)
y_test_transformed = le.transform(y_test)

In [51]:
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier()
model.fit(X_train_transformed,y_train_transformed)
y_predict = model.predict(X_test_transformed)

In [52]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test,y_predict)
accuracy

0.7536231884057971